In [10]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error

#
# =========================
# SETTINGS - HOUSE01 WITH-HISTORY
# =========================
REAL_DATA_PATH = r"C:\Plegma_Programming\PLEGMA_final_hourly_dataset.csv"
HOUSE_FOLDER = Path(r"C:\Plegma_Programming\Evaluation\House10")
HOME_ID = "home10"
REAL_DATE = "2023-09-29"
PRED_PATH = HOUSE_FOLDER / "ui_pred_combined_withhistory_athens_2026-09-29_20260406_195949.csv"


# =========================
# LOAD REAL DATA
# =========================
df_real = pd.read_csv(REAL_DATA_PATH)
df_real["timestamp"] = pd.to_datetime(df_real["timestamp"], errors="coerce")

real_day = pd.to_datetime(REAL_DATE).date()

# =========================
# BUILD REAL PROFILE (24h)
# =========================
real_profile = df_real[
    (df_real["home_id"] == HOME_ID) &
    (df_real["timestamp"].dt.date == real_day)
].copy()

real_profile = real_profile.sort_values("timestamp")
real_profile["hour"] = real_profile["timestamp"].dt.hour
real_profile = real_profile[["timestamp", "hour", "consumption_Wh"]].rename(
    columns={"consumption_Wh": "actual_Wh"}
)

print(f"Real profile rows for {HOME_ID} on {REAL_DATE}: {len(real_profile)}")

if len(real_profile) != 24:
    raise ValueError(f"Expected 24 rows in real profile, found {len(real_profile)}.")

# =========================
# LOAD UI PREDICTIONS
# =========================
if not PRED_PATH.exists():
    raise FileNotFoundError(f"Prediction file not found: {PRED_PATH}")

df_pred = pd.read_csv(PRED_PATH)
df_pred["timestamp"] = pd.to_datetime(df_pred["timestamp"], errors="coerce")
df_pred = df_pred.sort_values("timestamp").copy()
df_pred["hour"] = df_pred["timestamp"].dt.hour

required_pred_cols = ["hour", "rf_Wh", "xgb_Wh", "lgbm_Wh", "ensemble_Wh"]
missing_cols = [c for c in required_pred_cols if c not in df_pred.columns]
if missing_cols:
    raise ValueError(f"Missing columns in prediction file: {missing_cols}")

pred_profile = df_pred[required_pred_cols].copy()

print(f"Prediction rows: {len(pred_profile)}")
if len(pred_profile) != 24:
    raise ValueError(f"Expected 24 rows in prediction file, found {len(pred_profile)}.")

# =========================
# MERGE REAL + PRED
# =========================
comparison = pd.merge(real_profile, pred_profile, on="hour", how="inner").sort_values("hour")

print(f"Merged rows: {len(comparison)}")
if len(comparison) != 24:
    raise ValueError(f"Expected 24 merged rows, found {len(comparison)}.")

# =========================
# METRICS FUNCTION
# =========================
def calc_metrics(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    nonzero_mask = y_true != 0
    if nonzero_mask.sum() > 0:
        mape = np.mean(np.abs((y_true[nonzero_mask] - y_pred[nonzero_mask]) / y_true[nonzero_mask])) * 100
    else:
        mape = np.nan

    pred_total_kWh = y_pred.sum() / 1000.0
    real_total_kWh = y_true.sum() / 1000.0

    return mae, rmse, mape, pred_total_kWh, real_total_kWh

# =========================
# EVALUATE ALL MODELS
# =========================
metrics_rows = []

for col, model_name in [
    ("rf_Wh", "RF"),
    ("xgb_Wh", "XGB"),
    ("lgbm_Wh", "LGBM"),
    ("ensemble_Wh", "Ensemble")
]:
    mae, rmse, mape, pred_total_kWh, real_total_kWh = calc_metrics(
        comparison["actual_Wh"], comparison[col]
    )

    metrics_rows.append({
        "home_id": HOME_ID,
        "scenario": "with_history",
        "model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape,
        "pred_total_kWh": pred_total_kWh,
        "real_total_kWh": real_total_kWh
    })

metrics_df = pd.DataFrame(metrics_rows).sort_values("RMSE")

# =========================
# SAVE FILES
# =========================
comparison_to_save = comparison[[
    "timestamp", "hour", "actual_Wh", "rf_Wh", "xgb_Wh", "lgbm_Wh", "ensemble_Wh"
]].copy()

comparison_path = HOUSE_FOLDER / "with_history_comparison.csv"
metrics_path = HOUSE_FOLDER / "with_history_metrics.csv"

comparison_to_save.to_csv(comparison_path, index=False)
metrics_df.to_csv(metrics_path, index=False)

# =========================
# PRINT RESULTS
# =========================
print("\nSaved:")
print(comparison_path)
print(metrics_path)

print("\nMetrics:")
print(metrics_df.to_string(index=False))

Real profile rows for home10 on 2023-09-29: 24
Prediction rows: 24
Merged rows: 24

Saved:
C:\Plegma_Programming\Evaluation\House10\with_history_comparison.csv
C:\Plegma_Programming\Evaluation\House10\with_history_metrics.csv

Metrics:
home_id     scenario    model        MAE       RMSE      MAPE  pred_total_kWh  real_total_kWh
 home10 with_history       RF  95.134917 140.038023 31.854518        6.552496        6.372967
 home10 with_history Ensemble 109.963402 152.727865 38.173862        7.307251        6.372967
 home10 with_history     LGBM 111.788283 154.590483 38.583954        7.084432        6.372967
 home10 with_history      XGB 128.160822 168.963469 46.950622        8.040431        6.372967
